# Part 2 - 에이전트 시스템 평가 및 배포

도구들을 Langchain을 사용하여 패키징하고(agent.py 파일에서), 첫 번째 에이전트 버전으로 평가를 실행할 예정입니다!

## MLFlow 3를 사용한 에이전트 평가
에이전트를 생성했으니, 이제 성능을 측정하고 이전 버전과 비교할 방법이 필요합니다.

Databricks는 MLFlow 3로 이를 매우 쉽게 만들어줍니다. 자동으로 다음을 수행할 수 있습니다:

- 모든 에이전트의 입력/출력 추적
- 최종 사용자 피드백 캡처
- 커스텀 또는 합성 평가 데이터셋에 대해 에이전트 평가
- 비즈니스 전문가와 함께 레이블링된 데이터셋 구축
- 각 평가를 이전 평가와 비교
- 프로덕션에 배포된 후 평가 추적 및 관리

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/ai-agent/mlflow-evaluate-0.png?raw=true?raw=true" width="800px">

### 우리 에이전트의 구성:

- [**agent.py**]($./agent.py): 이 파일에서 Langchain을 사용하여 사용 가능한 에이전트를 준비했습니다.
- [**agent_config.yaml**]($./agent_config.yaml): 이 파일에는 시스템 프롬프트와 사용할 LLM 엔드포인트를 포함한 에이전트 설정이 담겨 있습니다.

자, 시작해서 이 노트북에서 Langchain 에이전트를 사용해 봅시다!


<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=601859075515029&notebook=%2F02-agent-eval%2F02.1_agent_evaluation&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F02-agent-eval%2F02.1_agent_evaluation&version=1">

In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

## 1/ 에이전트 빌드 및 등록

### 1.1/ 에이전트 설정 정의
먼저 설정 파일을 업데이트하여 langchain 에이전트가 사용할 도구, 기본 시스템 프롬프트, 사용할 엔드포인트를 지정합니다.

In [0]:
import yaml
import mlflow

rag_chain_config = {
    "config_version_name": "first_config",
    "input_example": [{"role": "user", "content": "john21@example.net 고객의 주문 내역을 확인해 주세요."}],
    "uc_tool_names": [f"{catalog}.{dbName}.*"],
    "system_prompt": "당신의 역할은 고객 지원입니다. 도구를 호출하여 답변을 제공하세요.",
    "llm_endpoint_name": LLM_ENDPOINT_NAME,
    "max_history_messages": 20,
    "retriever_config": None,
    "mcp_server_urls": [] 
}
try:
    with open('agent_config.yaml', 'w') as f:
        yaml.dump(rag_chain_config, f)
except:
    print('빌드 작업에서 작동하도록 패스')
model_config = mlflow.models.ModelConfig(development_config='agent_config.yaml')

`agent.py` 파일에서 langchain을 사용하여 AGENT를 생성했습니다. 파일을 탐색하여 내부 코드를 확인할 수 있습니다.

이 노트북에서는 간단하게 임포트하고 요청을 보내서 MLFlow Trace UI로 내부 추적을 탐색해 볼 겁니다:

In [0]:
from agent import AGENT

# 올바른 요청 형식
request_example = "john21@example.net에 대한 정보를 알려주세요."
answer = AGENT.predict({"input":[{"role": "user", "content": request_example}]})

### 1.2/ MLFlow 추적 확인

<img src="https://i.imgur.com/tNYUHdC.gif" style="float: right" width="700px">

오른쪽 노트북 메뉴에서 실험을 열어보세요. traces에서 방금 보낸 메시지를 확인할 수 있습니다: `john21@example.net에 대한 정보를 알려주세요.`.

MLFlow는 모든 입력/출력과 내부 추적을 기록하여 기존 요청을 분석하고 시간이 지남에 따라 더 나은 평가 데이터셋을 만들 수 있습니다!

MLFlow는 모든 에이전트 요청을 추적할 뿐만 아니라, 최종 사용자 피드백도 쉽게 캐폄하여 어떤 답변이 잘못되었는지 빠르게 감지하고 그에 따라 에이전트를 개선할 수 있습니다!

*애플리케이션을 배포할 때 피드백을 캐포하는 방법을 보여드리겠습니다!*

### 1.3/ `agent`를 MLflow 모델로 로깅

좋아 보입니다! 직렬화 문제를 피하기 위해 [agent]($./agent) 파이썬 파일을 사용하여 MLFlow 레지스트리에 에이전트를 로깅합니다. [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code)를 참고하세요.

*에이전트가 제대로 작동하는 데 필요한 Databricks 리소스(함수, warehouse 등) 목록도 전달합니다. 이는 배포 시 권한을 자동으로 처리해 줍니다!*

In [0]:
import mlflow
def log_customer_support_agent_model(resources, request_example):
    with mlflow.start_run(run_name=model_config.get('config_version_name')):
        return mlflow.pyfunc.log_model(
            name="agent",
            python_model="agent.py",
            model_config="agent_config.yaml",
            input_example={"input": [{"role": "user", "content": request_example}]},
            resources=resources, # 배포 시 자동 인증 패스스루를 위해 지정할 Databricks 리소스(엔드포인트, 함수, vs...) 결정
            extra_pip_requirements=["databricks-connect"]
        )
logged_agent_info = log_customer_support_agent_model(AGENT.get_resources(), request_example)

### 1.4/ 모델을 로드하고 테스트하기
모델이 MLFlow에 저장되었습니다! 로드해서 사용해 봅시다. 최종 답변을 더 쉽게 추출할 수 있도록, 그리고 평가를 더 쉽게 할 수 있도록 predict 함수를 래핑하겠습니다:

In [0]:
import pandas as pd
# 모델을 로드하고 예측 함수 생성
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")
def predict_wrapper(question):
    # 챗 스타일 모델용 형식
    model_input = pd.DataFrame({
        "input": [[{"role": "user", "content": question}]]
    })
    response = loaded_model.predict(model_input)
    return response['output'][-1]['content'][-1]['text']

answer = predict_wrapper("john21@example.net에 대한 정보를 알려주세요.")
print(answer)

## 2/ 평가

### 2.1/ [Agent Evaluation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)을 사용하여 에이전트 평가

이 데모의 일부로 사용할 준비가 된 평가 데이터셋을 준비했습니다. 하지만 여러 전략이 존재합니다:

- 자체 평가 데이터셋 생성 (우리가 할 것)
- MLFlow의 기존 trace를 사용하여 데이터셋에 추가 (API 또는 실험 UI에서)
- Databricks genai eval 합성 데이터셋 생성 사용 (예시는 [PDF RAG 노트북]($../03-knowledge-base-rag/03.1-pdf-rag-tool) 참고)
- MLFlow UI를 직접 사용하여 전문가로부터 인사이트를 얻을 수 있는 레이블링 세션 생성!

참고: trace에서 기존 LLM 호출을 선택하여 UI로 평가 데이터셋에 추가하거나 API를 직접 사용할 수도 있습니다:

```
traces = mlflow.search_traces(filter_string=f"attributes.timestamp_ms > {ten_minutes_ago} AND attributes.status = 'OK'", order_by=["attributes.timestamp_ms DESC"])
```

In [0]:
%run ../_resources/04-eval-dataset-generation

In [0]:
eval_example = spark.read.json(f"/Volumes/{catalog}/{dbName}/{volume_name}/eval_dataset")
display(eval_example)

### 2.2/ MLFlow 데이터셋 생성
API를 사용하여 데이터셋을 생성합니다. 실험 UI에서 직접 수행할 수도 있습니다!

In [0]:
import mlflow
import mlflow.genai.datasets

eval_dataset_table_name = f"{catalog}.{dbName}.ai_agent_mlflow_eval"

try:
  eval_dataset = mlflow.genai.datasets.get_dataset(eval_dataset_table_name)
except Exception as e:
  if 'does not exist' in str(e):
    eval_dataset = mlflow.genai.datasets.create_dataset(eval_dataset_table_name)
    # 평가 데이터셋에 예제 추가
    eval_dataset.merge_records(eval_example)
    print("평가 데이터셋에 레코드를 추가했습니다.")

# 데이터셋 미리보기
display(eval_dataset.to_df())

## 2.3/ 에이전트 동작을 추적하기 위한 가이드라인 추가

<img src="https://i.imgur.com/M3kLBHF.gif" style="float:right" width="700px">

MLFlow 3.0은 에이전트 동작을 평가하기 위한 커스텀 가이드라인을 생성할 수 있게 해줍니다.

기본 제공되는 것 몇 가지를 사용하고, 단계와 추론에 대한 커스텀 `Guidelines`를 추가하겠습니다: LLM이 내부 도구를 언급하지 않고 답변을 출력하길 원합니다.

In [0]:
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Safety, Guidelines

def get_scorers():
    return [
        RetrievalGroundedness(),  # 이메일 콘텐츠가 검색된 데이터에 기반하는지 확인
        RelevanceToQuery(),  # 이메일이 사용자 요청을 다루는지 확인
        Safety(),  # 유해하거나 부적절한 콘텐츠 확인
        Guidelines(
            guidelines="""
            답변은 추론을 보여주지 않고 작성되어야 합니다.
            - 무언가를 찾아야 한다고 언급하지 마세요
            - 도구나 사용한 함수를 언급하지 마세요
            - 중간 단계나 추론을 말하지 마세요
            """,
            name="steps_and_reasoning",
        )
    ]
    
scorers = get_scorers()

## 2.4/ 가이드라인에 따른 평가 실행

이제 가이드라인을 사용하여 데이터셋을 평가해 보겠습니다:

In [0]:
with mlflow.start_run(run_name='eval_with_no_reasoning_instructions'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

## 3/ 더 나은 시스템 프롬프트로 평가 메트릭 개선

평가에서 볼 수 있듯이, 에이전트가 내부 도구와 단계에 대한 많은 정보를 출력합니다.
예를 들어, 다음과 같이 언급합니다:

`먼저, 고객님의 이메일 주소를 확인하여 고객 기록을 조회해야 합니다. 실례지만, 정확한 도움을 위해 이메일 주소를 말씀해 주세요.`

좋은 추론이지만, 최종 답변에서는 이렇게 나오면 안 됩니다!

### 3.1/ 더 나은 시스템 프롬프트로 새 모델 버전 배포

이러한 동작을 피하기 위해 더 나은 지침으로 시스템 프롬프트를 업데이트하고, 평가를 실행하여 개선되었는지 확인해 봅시다!

In [0]:
try:
    config = yaml.safe_load(open("agent_config.yaml"))
    config["config_version_name"] = "better_prompt"
    config["system_prompt"] = (
        "당신은 통신사 상담원입니다. 요금, 지원 서비스, 계정 정보 등 고객의 요청 내용에 대해 적합한 도구를 실행하여 도움을 제공하세요. "
        "답변 시 내부 도구 명칭이나 추론 과정을 언급해서는 안 되며, '기록에 따르면'과 같은 조회 중임을 암시하는 표현도 삼가시기 바랍니다."
    )
    yaml.dump(config, open("agent_config.yaml", "w"))
except Exception as e:
    print(f"Skipped update - ignore for job run - {e}")

In [0]:
# 새 프롬프트를 캡처하기 위해 MLflow에 에이전트를 다시 로깅합니다
logged_agent_info = log_customer_support_agent_model(AGENT.get_resources(), request_example)

In [0]:
# 평가에서 `predict_wrapper`를 통해 사용할 모델 로드
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")

with mlflow.start_run(run_name='eval_with_reasoning_instructions'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

실험을 열고 결과를 확인하세요!

이전 실행과 현재 실행을 선택하여 비교해 보세요. 개선된 부분을 확인할 수 있을 겁니다!

## 4/ 에이전트를 엔드포인트로 배포!

모든 것이 잘 보입니다! 최신 버전이 괜찮은 평가 점수를 받았습니다. 최종 사용자 챗 애플리케이션을 위해 실시간 엔드포인트로 배포하겠습니다.

### 4.1/ Unity Catalog에 새 모델 버전 등록


In [0]:
from mlflow import MlflowClient
UC_MODEL_NAME = f"{catalog}.{dbName}.{MODEL_NAME}"

# UC에 모델 등록
client = MlflowClient()
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME, tags={"model": "customer_support_agent"})
client.set_registered_model_alias(name=UC_MODEL_NAME, alias="model-to-deploy", version=uc_registered_model_info.version)

# 생성된 에이전트에 대한 HTML 링크 생성
displayHTML(f'<a href="/explore/data/models/{catalog}/{dbName}/{MODEL_NAME}" target="_blank">등록된 에이전트를 보려면 Unity Catalog 열기</a>')

### 4.2/ 에이전트 배포

이제 모델 엔드포인트를 시작하겠습니다:

In [0]:

from databricks import agents

try:
    # 모델을 리뷰 앱과 모델 서빙 엔드포인트에 배포
    if len(agents.get_deployments(model_name=UC_MODEL_NAME, model_version=uc_registered_model_info.version)) == 0:
        agents.deploy(
            UC_MODEL_NAME, 
            uc_registered_model_info.version, 
            endpoint_name=ENDPOINT_NAME, 
            scale_to_zero=True, ## Free Edtion의 경우 필수로 활성화
            tags={"project": "dbdemos"}
        )
        print(f"배포 성공: {ENDPOINT_NAME}")
    else:
        print(f"이미 배포되어 있습니다: {ENDPOINT_NAME}")
except Exception as e:
    if "Scale to zero must be enabled" in str(e):
        print("⚠️  배포 건너뜀: 이 워크스페이스에는 scale-to-zero 기능이 필요합니다.")
        print("워크스페이스 관리자에게 scale-to-zero 활성화를 요청하거나,")
        print("노트북에서 직접 모델을 로드하여 사용하실 수 있습니다.")
        print(f"\n모델은 Unity Catalog에 등록되어 있습니다: {UC_MODEL_NAME} (버전 {uc_registered_model_info.version})")
    else:
        raise e

## 다음: 지식 기반에 대한 질문에 답변할 도구 추가 (RAG + PDF에 대한 Vector Search)

모델이 잘 작동하고 있지만, 고객 지원팀이 구독에 대해 가질 수 있는 특정 질문에는 답할 수 없습니다.

예를 들어, WIFI 라우터의 특정 오류 코드를 해결하는 방법을 에이전트에게 물어보면, 관련 정보가 없기 때문에 실패할 것입니다.

[03-knowledge-base-rag/03.1-pdf-rag-tool]($../03-knowledge-base-rag/03.1-pdf-rag-tool)을 열어보세요